# Phase 3B: Cleaned Filtered Dataset

This Colab notebook creates `cleaned-filtered-v1` for Phase 3 of the Llama 3.2-1B training plan.

**Purpose:** Test whether removing obvious low-value scraped pages creates a cleaner adaptation corpus.

The notebook loads `sapinsapin/halohalo`, normalizes text, applies this variant's cleaning rules, rebuilds train/validation splits, writes dataset artifacts, and proves the result with executable validation checks.

## Runtime Notes

Use a standard Colab Python runtime. A GPU is not required because this notebook only cleans and validates data.

Artifacts are copied to Google Drive under:

`/content/drive/MyDrive/PH-SapinSapin/llama-3.2-1B/datasets/<variant-name>/`

In [1]:
!pip -q install -U "datasets>=2.20.0" "pandas>=2.0.0" "pyarrow>=15.0.0"

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
import hashlib
import json
import math
import os
import random
import re
import shutil
import unicodedata

import pandas as pd
from datasets import Dataset, concatenate_datasets, load_dataset
from IPython.display import Markdown, display

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DATASET_NAME = "sapinsapin/halohalo"
SEED = 3407
VALIDATION_FRACTION = 0.02
MIN_ROWS_FOR_VALIDATION_LANGUAGE = 20
LOCAL_RESULTS_ROOT = Path("/Users/zeraphim/Documents/Files/ThisIsZeraphim/Open Source/PH-SapinSapin/DataCleaning/results")
if IN_COLAB:
    LOCAL_ROOT = Path("/content/phase3_cleaned_datasets")
    DRIVE_ROOT = Path("/content/drive/MyDrive/PH-SapinSapin/llama-3.2-1B/datasets")
else:
    LOCAL_ROOT = LOCAL_RESULTS_ROOT / "_working"
    DRIVE_ROOT = LOCAL_RESULTS_ROOT

REQUIRED_COLUMNS = [
    "id",
    "text",
    "url",
    "date",
    "dump",
    "file_path",
    "detected_lang",
    "word_count",
    "title",
    "source",
    "language",
    "token_count",
    "content_hash",
    "crawled_at",
]

TARGET_LANGUAGES = ["fil", "hil", "tgl", "bcl", "ilo", "ceb"]
DETECTED_LANGUAGE_MAP = {"tl": "tgl", "tgl": "tgl", "hil": "hil", "bik": "bcl", "bcl": "bcl", "ceb": "ceb", "ilo": "ilo"}
TRUSTED_MISSING_DETECTED_SOURCES = {
    "transcribed",
    "filnet_novels",
    "palito",
    "newspaper_fil",
    "proj_gutenberg",
    "filipiniana",
    "isip_ilk",
    "isip_ceb",
}

NOISE_URL_PATTERNS = [
    r"/login\\b",
    r"/log-in\\b",
    r"/signin\\b",
    r"/sign-in\\b",
    r"/edit\\b",
    r"action=edit",
    r"/feed/?$",
    r"/feed/",
    r"/tag/",
    r"/tags/",
    r"/archive/",
    r"/category/",
    r"/author/",
    r"/search",
    r"/slideshow",
    r"/slideshare",
    r"/wp-login",
    r"/special:",
    r"/wiki/special:",
    r"/w/index\\.php",
]

NOISE_TEXT_PATTERNS = [
    r"this content was originally published by",
    r"researchpdf available",
    r"pdf available",
    r"login",
    r"sign in",
    r"create account",
    r"forgot password",
    r"slideshow",
]

BLOCKED_DOMAINS = {
    "www.researchgate.net",
    "researchgate.net",
    "www.slideshare.net",
    "slideshare.net",
    "auth.wikimedia.org",
    "commons.wikimedia.org",
    "meta.wikimedia.org",
}

ALLOWED_WIKI_DOMAINS = {
    "tl.wikipedia.org",
    "tl.m.wikipedia.org",
    "ceb.wikipedia.org",
}

UTILITY_WIKI_PREFIXES = (
    "special:",
    "talk:",
    "user:",
    "user_talk:",
    "template:",
    "template_talk:",
    "category:",
    "category_talk:",
    "file:",
    "file_talk:",
    "help:",
    "portal:",
    "wikipedia:",
)

STRICT_MAX_TOKENS = 4096
BALANCE_TOKEN_CAP_MULTIPLIER = 1.25

VARIANT_NAME = 'cleaned-filtered-v1'
VARIANT_LABEL = 'Deduplicated + noise filtered'
VARIANT_PURPOSE = 'Test whether removing obvious low-value scraped pages creates a cleaner adaptation corpus.'
APPLY_NOISE_FILTER = True
APPLY_BALANCING = False
APPLY_HIGH_CONFIDENCE = False

In [4]:
if IN_COLAB:
    drive.mount("/content/drive")
else:
    print(f"Not running inside Colab; artifacts will be written locally under {DRIVE_ROOT}.")

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = LOCAL_ROOT / VARIANT_NAME
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)


def normalize_text(value: object) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("\x00", " ")
    text = "\n".join(line.rstrip() for line in text.splitlines())
    return text.strip()


def stable_content_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def token_count(row: dict) -> int:
    value = row.get("token_count")
    if value is None:
        return max(1, len((row.get("text") or "").split()))
    try:
        return int(value)
    except Exception:
        return max(1, len((row.get("text") or "").split()))


def load_raw_rows() -> list[dict]:
    loaded = load_dataset(DATASET_NAME)
    splits = [loaded[name] for name in loaded.keys()]
    dataset = concatenate_datasets(splits) if len(splits) > 1 else splits[0]
    missing = sorted(set(REQUIRED_COLUMNS) - set(dataset.column_names))
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")
    return [dict(row) for row in dataset]


def prepare_rows(rows: list[dict]) -> tuple[list[dict], Counter]:
    counters = Counter()
    prepared: list[dict] = []
    for original_index, row in enumerate(rows):
        item = dict(row)
        item["_original_index"] = original_index
        item["text"] = normalize_text(item.get("text"))
        if not item["text"]:
            counters["empty_text"] += 1
            continue
        if not item.get("content_hash"):
            item["content_hash"] = stable_content_hash(item["text"])
            counters["content_hash_filled"] += 1
        item["token_count"] = token_count(item)
        prepared.append(item)
    return prepared, counters


def deduplicate_by_hash(rows: list[dict]) -> tuple[list[dict], Counter, list[dict]]:
    seen: set[str] = set()
    kept: list[dict] = []
    removed_examples: list[dict] = []
    counters = Counter()
    for row in sorted(rows, key=lambda item: (str(item.get("content_hash")), int(item.get("_original_index", 0)))):
        content_hash = row["content_hash"]
        if content_hash in seen:
            counters["duplicate_content_hash"] += 1
            if len(removed_examples) < 12:
                removed_examples.append(example_row(row, "duplicate_content_hash"))
            continue
        seen.add(content_hash)
        kept.append(row)
    kept.sort(key=lambda item: int(item.get("_original_index", 0)))
    return kept, counters, removed_examples


def compile_patterns(patterns: list[str]) -> list[re.Pattern]:
    return [re.compile(pattern, re.I) for pattern in patterns]


URL_PATTERNS = compile_patterns(NOISE_URL_PATTERNS)
TEXT_PATTERNS = compile_patterns(NOISE_TEXT_PATTERNS)


def parsed_domain(row: dict) -> str:
    url = row.get("url") or ""
    return urlparse(url).netloc.lower()


def wiki_page_title_from_url(url: str) -> str:
    parsed = urlparse(url)
    path = parsed.path.lower()
    if "/wiki/" not in path:
        return ""
    return path.split("/wiki/", 1)[1]


def noise_reasons(row: dict) -> list[str]:
    reasons: list[str] = []
    url = (row.get("url") or "").lower()
    title = (row.get("title") or "").lower()
    text_head = (row.get("text") or "")[:1000].lower()
    domain = parsed_domain(row)

    if domain in BLOCKED_DOMAINS:
        reasons.append(f"blocked_domain:{domain}")

    if "wikipedia.org" in domain or "wikimedia.org" in domain or "wikiquote.org" in domain or "wiktionary.org" in domain:
        if domain not in ALLOWED_WIKI_DOMAINS:
            reasons.append(f"non_target_wiki_domain:{domain}")
        wiki_title = wiki_page_title_from_url(url)
        if wiki_title.startswith(UTILITY_WIKI_PREFIXES):
            reasons.append("wiki_utility_page")

    for pattern in URL_PATTERNS:
        if pattern.search(url):
            reasons.append(f"url_pattern:{pattern.pattern}")

    blob = " ".join([title, text_head, url])
    for pattern in TEXT_PATTERNS:
        if pattern.search(blob):
            reasons.append(f"text_pattern:{pattern.pattern}")

    source = (row.get("source") or "").lower()
    if "researchgate" in blob or "researchgate" in source:
        reasons.append("researchgate_signal")
    if "slideshare" in blob or "slideshare" in source:
        reasons.append("slideshare_signal")

    return sorted(set(reasons))


def hard_block_reasons(row: dict) -> list[str]:
    """Deterministic blockers used for the final zero-remaining-noise assertion."""
    hard_reasons = []
    for reason in noise_reasons(row):
        if reason.startswith(("blocked_domain:", "non_target_wiki_domain:", "url_pattern:")):
            hard_reasons.append(reason)
        elif reason in {"wiki_utility_page", "researchgate_signal", "slideshare_signal"}:
            hard_reasons.append(reason)
        elif "this content was originally published by" in reason or "pdf available" in reason:
            hard_reasons.append(reason)
    return hard_reasons


def apply_noise_filter(rows: list[dict]) -> tuple[list[dict], Counter, list[dict]]:
    kept: list[dict] = []
    counters = Counter()
    removed_examples: list[dict] = []
    for row in rows:
        reasons = noise_reasons(row)
        if reasons:
            counters.update(reasons)
            counters["noise_filtered_total"] += 1
            if len(removed_examples) < 20:
                removed_examples.append(example_row(row, "; ".join(reasons)))
            continue
        kept.append(row)
    return kept, counters, removed_examples


def detected_language_matches(row: dict) -> bool:
    detected = row.get("detected_lang")
    language = row.get("language")
    source = row.get("source")
    if detected:
        return DETECTED_LANGUAGE_MAP.get(str(detected).lower(), str(detected).lower()) == language
    return source in TRUSTED_MISSING_DETECTED_SOURCES


def high_confidence_reasons(row: dict) -> list[str]:
    reasons: list[str] = []
    if not detected_language_matches(row):
        reasons.append("language_confidence_failed")
    if token_count(row) > STRICT_MAX_TOKENS:
        reasons.append("above_strict_max_tokens")
    if token_count(row) <= 0:
        reasons.append("invalid_token_count")
    return reasons


def apply_high_confidence_filter(rows: list[dict]) -> tuple[list[dict], Counter, list[dict]]:
    kept: list[dict] = []
    counters = Counter()
    removed_examples: list[dict] = []
    for row in rows:
        reasons = high_confidence_reasons(row)
        if reasons:
            counters.update(reasons)
            counters["high_confidence_filtered_total"] += 1
            if len(removed_examples) < 20:
                removed_examples.append(example_row(row, "; ".join(reasons)))
            continue
        kept.append(row)
    return kept, counters, removed_examples


def balance_by_language_token_budget(rows: list[dict]) -> tuple[list[dict], dict, list[dict]]:
    rng = random.Random(SEED)
    by_language: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        by_language[row.get("language") or "unknown"].append(row)

    token_totals = {language: sum(token_count(row) for row in group) for language, group in by_language.items()}
    nonzero_totals = [total for total in token_totals.values() if total > 0]
    median_tokens = int(pd.Series(nonzero_totals).median()) if nonzero_totals else 0
    cap = max(1, int(median_tokens * BALANCE_TOKEN_CAP_MULTIPLIER))

    kept: list[dict] = []
    removed: list[dict] = []
    balance_details = {
        "token_cap": cap,
        "before_token_totals": token_totals,
        "after_token_totals": {},
        "strategy": "Keep all languages at or below cap; deterministically sample dominant languages until their token budget is near the cap.",
    }

    for language, group in sorted(by_language.items()):
        group = list(group)
        total = token_totals[language]
        if total <= cap:
            kept.extend(group)
            continue

        rng.shuffle(group)
        selected: list[dict] = []
        selected_tokens = 0
        for row in group:
            row_tokens = token_count(row)
            if selected and selected_tokens + row_tokens > cap:
                removed.append(example_row(row, f"language_balance_cap:{language}"))
                continue
            selected.append(row)
            selected_tokens += row_tokens
        kept.extend(selected)

    kept.sort(key=lambda item: int(item.get("_original_index", 0)))
    balance_details["after_token_totals"] = dict(Counter({lang: 0 for lang in by_language}))
    after_counter = Counter()
    for row in kept:
        after_counter[row.get("language") or "unknown"] += token_count(row)
    balance_details["after_token_totals"] = dict(after_counter)
    return kept, balance_details, removed[:20]


def language_stats(rows: list[dict]) -> pd.DataFrame:
    row_counts = Counter(row.get("language") or "unknown" for row in rows)
    token_counts = Counter()
    total_rows = len(rows)
    total_tokens = sum(token_count(row) for row in rows)
    for row in rows:
        token_counts[row.get("language") or "unknown"] += token_count(row)
    records = []
    for language in sorted(row_counts, key=lambda key: (-token_counts[key], key)):
        records.append({
            "language": language,
            "rows": row_counts[language],
            "row_share_pct": round(100 * row_counts[language] / total_rows, 2) if total_rows else 0,
            "tokens": token_counts[language],
            "token_share_pct": round(100 * token_counts[language] / total_tokens, 2) if total_tokens else 0,
        })
    return pd.DataFrame(records)


def corpus_summary(rows: list[dict]) -> dict:
    hashes = [row.get("content_hash") for row in rows]
    duplicate_extra = len(hashes) - len(set(hashes))
    noise_hits = sum(1 for row in rows if noise_reasons(row))
    lang_match_rows = sum(1 for row in rows if detected_language_matches(row))
    return {
        "rows": len(rows),
        "tokens": int(sum(token_count(row) for row in rows)),
        "duplicate_extra_rows": int(duplicate_extra),
        "noise_hit_rows": int(noise_hits),
        "language_confidence_pass_rows": int(lang_match_rows),
        "language_confidence_pass_pct": round(100 * lang_match_rows / len(rows), 2) if rows else 0,
        "max_token_count": max([token_count(row) for row in rows], default=0),
    }


def imbalance_score(rows: list[dict]) -> float:
    stats = language_stats(rows)
    if stats.empty:
        return 0.0
    shares = stats["token_share_pct"].tolist()
    return round(max(shares) - min(shares), 4)


def split_rows(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    rng = random.Random(SEED)
    train: list[dict] = []
    validation: list[dict] = []
    by_language: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        by_language[row.get("language") or "unknown"].append(row)

    for language, group in sorted(by_language.items()):
        group = list(group)
        rng.shuffle(group)
        if len(group) >= MIN_ROWS_FOR_VALIDATION_LANGUAGE:
            val_count = max(1, int(round(len(group) * VALIDATION_FRACTION)))
        else:
            val_count = 0
        validation.extend(group[:val_count])
        train.extend(group[val_count:])

    train.sort(key=lambda item: int(item.get("_original_index", 0)))
    validation.sort(key=lambda item: int(item.get("_original_index", 0)))
    return train, validation


def strip_internal_columns(row: dict) -> dict:
    return {key: row.get(key) for key in REQUIRED_COLUMNS}


def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(strip_internal_columns(row), ensure_ascii=False) + "\n")


def example_row(row: dict, reason: str) -> dict:
    return {
        "reason": reason,
        "language": row.get("language"),
        "source": row.get("source"),
        "token_count": token_count(row),
        "title": (row.get("title") or "")[:120],
        "url": (row.get("url") or "")[:180],
    }


def display_summary(title: str, summary: dict) -> None:
    display(Markdown(f"### {title}"))
    display(pd.DataFrame([summary]))

Not running inside Colab; artifacts will be written locally under /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results.


## 1. Load, Clean, And Summarize

In [5]:
raw_rows = load_raw_rows()
prepared_rows, preparation_counts = prepare_rows(raw_rows)

dedup_rows, dedup_counts, dedup_examples = deduplicate_by_hash(prepared_rows)
current_rows = dedup_rows
removed_examples = {"deduplication": dedup_examples}
removal_counts = Counter(preparation_counts)
removal_counts.update(dedup_counts)
balance_details = None

variant_b_reference_rows = None
if APPLY_NOISE_FILTER or APPLY_BALANCING or APPLY_HIGH_CONFIDENCE:
    filtered_rows, noise_counts, noise_examples = apply_noise_filter(current_rows)
    current_rows = filtered_rows
    removal_counts.update(noise_counts)
    removed_examples["noise_filter"] = noise_examples
    variant_b_reference_rows = filtered_rows

if APPLY_BALANCING:
    before_balance_rows = list(current_rows)
    balanced_rows, balance_details, balance_examples = balance_by_language_token_budget(current_rows)
    current_rows = balanced_rows
    removal_counts["language_balance_removed_total"] = len(before_balance_rows) - len(balanced_rows)
    removed_examples["language_balance"] = balance_examples

if APPLY_HIGH_CONFIDENCE:
    before_high_confidence_rows = list(current_rows)
    high_confidence_rows, high_confidence_counts, high_confidence_examples = apply_high_confidence_filter(current_rows)
    current_rows = high_confidence_rows
    removal_counts.update(high_confidence_counts)
    removed_examples["high_confidence"] = high_confidence_examples

train_rows, validation_rows = split_rows(current_rows)
cleaned_rows = train_rows + validation_rows

before_summary = corpus_summary(prepared_rows)
after_dedup_summary = corpus_summary(dedup_rows)
after_summary = corpus_summary(cleaned_rows)
train_summary = corpus_summary(train_rows)
validation_summary = corpus_summary(validation_rows)

display_summary("Raw Prepared Corpus", before_summary)
display_summary("After Deduplication", after_dedup_summary)
display_summary("Final Cleaned Corpus", after_summary)
display_summary("Train Split", train_summary)
display_summary("Validation Split", validation_summary)

display(Markdown("### Language Distribution Before Cleaning"))
display(language_stats(prepared_rows))
display(Markdown("### Language Distribution After Cleaning"))
display(language_stats(cleaned_rows))
display(Markdown("### Removal Counts By Rule"))
display(pd.DataFrame(removal_counts.most_common(), columns=["rule", "count"]))

for section, examples in removed_examples.items():
    display(Markdown(f"### Representative Removed Examples: {section}"))
    display(pd.DataFrame(examples if examples else [{"note": "No examples for this section."}]))

if balance_details:
    display(Markdown("### Language Balance Details"))
    display(pd.DataFrame([balance_details]))

### Raw Prepared Corpus

,rows,tokens,duplicate_extra_rows,noise_hit_rows,language_confidence_pass_rows,language_confidence_pass_pct,max_token_count
0,45536,26298857,10504,10216,37319,81.95,89509


### After Deduplication

,rows,tokens,duplicate_extra_rows,noise_hit_rows,language_confidence_pass_rows,language_confidence_pass_pct,max_token_count
0,35032,13323393,0,4467,31365,89.53,89509


### Final Cleaned Corpus

...

,rule,count
0,duplicate_content_hash,10504
1,noise_filtered_total,4467
2,text_pattern:this content was originally publi...,2152
3,researchgate_signal,732
4,blocked_domain:www.researchgate.net,727
...,...,...
62,non_target_wiki_domain:fiu-vro.wikipedia.org,1
63,non_target_wiki_domain:bi.wikipedia.org,1
64,non_target_wiki_domain:dag.wikipedia.org,1
65,non_target_wiki_domain:ang.wikipedia.org,1


### Representative Removed Examples: deduplication

,reason,language,source,token_count,title,url
0,duplicate_content_hash,hil,halo-hil,584,Tulfo: Batas sa PWD rebisahin | Pilipino Star ...,https://www.philstar.com/pilipino-star-ngayon/...
1,duplicate_content_hash,hil,halo-hil,584,Tulfo: Batas sa PWD rebisahin | Pilipino Star ...,https://www.philstar.com/pilipino-star-ngayon/...
2,duplicate_content_hash,hil,halo-hil,584,Tulfo: Batas sa PWD rebisahin | Pilipino Star ...,https://www.philstar.com/pilipino-star-ngayon/...
3,duplicate_content_hash,hil,halo-hil,584,Tulfo: Batas sa PWD rebisahin | Pilipino Star ...,https://www.philstar.com/pilipino-star-ngayon/...
4,duplicate_content_hash,tgl,halo-tgl,756,"Sining - Wikipedia, ang malayang ensiklopedya",https://tl.wikipedia.org/wiki/Sining
5,duplicate_content_hash,bcl,halo-bcl,1280,Solo trip | Magbikol Kita,https://magbikolkita.com/layason-officer/solo-...
6,duplicate_content_hash,hil,halo-hil,447,"Major rehab sa MRT-3, sisimulan sa susunod na ...",https://www.philstar.com/pilipino-star-ngayon/...
7,duplicate_content_hash,hil,halo-hil,906,"Tom, kaswal na sa ‘unmarry’ nila ni Carla | Pi...",https://www.philstar.com/pilipino-star-ngayon/...
8,duplicate_content_hash,hil,halo-hil,59,Just a moment...,https://www.researchgate.net/publication/39112...
9,duplicate_content_hash,tgl,halo-tgl,64,"Gumawa ng account - Wikipedia, ang malayang en...",https://tl.wikipedia.org/w/index.php?title=Nat...


### Representative Removed Examples: noise_filter

,reason,language,source,token_count,title,url
0,blocked_domain:www.researchgate.net; researchg...,hil,halo-hil,80,Catabian's research works,https://www.researchgate.net/scientific-contri...
1,text_pattern:this content was originally publi...,hil,halo-hil,1119,"Retired AFP at PNP officers, nagpapansin kay B...",https://www.philstar.com/punto-mo/2024/01/05/2...
2,text_pattern:this content was originally publi...,hil,halo-hil,479,Nakalimutan at nakaligtaan | Pilipino Star Ngayon,https://www.philstar.com/opinyon/2025/11/16/24...
3,text_pattern:this content was originally publi...,hil,halo-hil,805,"Manilyn, hindi sinanay na may kasambahay ang m...",https://www.philstar.com/pilipino-star-ngayon/...
4,text_pattern:this content was originally publi...,hil,halo-hil,1133,"Krystel Go tinalo sina Angelica at Nadine, Bes...",https://www.philstar.com/pilipino-star-ngayon/...
5,text_pattern:this content was originally publi...,hil,halo-hil,775,"Indiscriminate firing ng PNP o sibilyan, no se...",https://www.philstar.com/punto-mo/2025/12/17/2...
6,url_pattern:/tags/,hil,halo-hil,1496,Apol - Tags | Philstar.com,https://www.philstar.com/tags/apol
7,text_pattern:this content was originally publi...,hil,halo-hil,435,Bubot na Bayabas (71) | Pilipino Star Ngayon,https://www.philstar.com/true-confessions/2025...
8,url_pattern:/tags/,hil,halo-hil,1923,duterte - Tags | Philstar.com,https://www.philstar.com/tags/duterte
9,url_pattern:/category/,hil,halo-hil,625,Local News Archives - Bombo Radyo Iloilo,https://iloilo.bomboradyo.com/category/top-sto...


## 2. Validate Cleaning Objectives

In [6]:
def validation_language_coverage_ok(train: list[dict], validation: list[dict], full: list[dict]) -> tuple[bool, list[str]]:
    train_langs = {row.get("language") for row in train}
    validation_langs = {row.get("language") for row in validation}
    full_counts = Counter(row.get("language") for row in full)
    missing = sorted(
        language
        for language, count in full_counts.items()
        if count >= MIN_ROWS_FOR_VALIDATION_LANGUAGE and (language not in train_langs or language not in validation_langs)
    )
    return not missing, missing


validation_results = []

def record_check(name: str, passed: bool, detail: str) -> None:
    validation_results.append({"check": name, "passed": bool(passed), "detail": detail})
    assert passed, f"{name} failed: {detail}"


cleaned_hashes = [row["content_hash"] for row in cleaned_rows]
train_hashes = {row["content_hash"] for row in train_rows}
validation_hashes = {row["content_hash"] for row in validation_rows}
required_missing_after = sorted(set(REQUIRED_COLUMNS) - set(strip_internal_columns(cleaned_rows[0]).keys())) if cleaned_rows else REQUIRED_COLUMNS
coverage_ok, missing_coverage_languages = validation_language_coverage_ok(train_rows, validation_rows, cleaned_rows)

record_check("cleaned_dataset_has_rows", len(cleaned_rows) > 0, f"{len(cleaned_rows):,} cleaned rows")
record_check("all_text_non_empty", all(bool(row.get("text")) for row in cleaned_rows), "all cleaned rows have non-empty text")
record_check("required_columns_preserved", not required_missing_after, f"missing columns: {required_missing_after}")
record_check("duplicate_content_hash_zero", len(cleaned_hashes) == len(set(cleaned_hashes)), f"duplicates: {len(cleaned_hashes) - len(set(cleaned_hashes))}")
record_check("train_validation_hash_overlap_zero", not (train_hashes & validation_hashes), f"overlap: {len(train_hashes & validation_hashes)}")
record_check("validation_language_coverage", coverage_ok, f"missing languages with enough rows: {missing_coverage_languages}")

if APPLY_NOISE_FILTER:
    raw_noise_hits = sum(1 for row in dedup_rows if noise_reasons(row))
    cleaned_hard_noise_hits = sum(1 for row in cleaned_rows if hard_block_reasons(row))
    record_check("noise_filter_removed_rows", removal_counts["noise_filtered_total"] > 0 if raw_noise_hits else True, f"removed {removal_counts['noise_filtered_total']} of {raw_noise_hits} noise-hit rows")
    record_check("hard_noise_patterns_absent", cleaned_hard_noise_hits == 0, f"remaining hard-block rows: {cleaned_hard_noise_hits}")

if APPLY_BALANCING:
    before_balance_score = imbalance_score(variant_b_reference_rows)
    after_balance_score = imbalance_score(cleaned_rows)
    record_check("language_token_balance_improved", after_balance_score < before_balance_score, f"before score {before_balance_score}, after score {after_balance_score}")

if APPLY_HIGH_CONFIDENCE:
    reference_summary = corpus_summary(variant_b_reference_rows)
    high_conf_summary = corpus_summary(cleaned_rows)
    record_check("high_confidence_smaller_than_filtered", high_conf_summary["rows"] < reference_summary["rows"], f"{high_conf_summary['rows']} < {reference_summary['rows']}")
    record_check("high_confidence_language_metric_improved", high_conf_summary["language_confidence_pass_pct"] >= reference_summary["language_confidence_pass_pct"], f"{high_conf_summary['language_confidence_pass_pct']} >= {reference_summary['language_confidence_pass_pct']}")
    record_check("strict_length_cap_met", high_conf_summary["max_token_count"] <= STRICT_MAX_TOKENS, f"max token count {high_conf_summary['max_token_count']}")

display(Markdown("### Validation Checklist"))
display(pd.DataFrame(validation_results))
display(Markdown("✅ All validation checks passed for this Phase 3 variant."))

### Validation Checklist

,check,passed,detail
0,cleaned_dataset_has_rows,True,"30,565 cleaned rows"
1,all_text_non_empty,True,all cleaned rows have non-empty text
2,required_columns_preserved,True,missing columns: []
3,duplicate_content_hash_zero,True,duplicates: 0
4,train_validation_hash_overlap_zero,True,overlap: 0
5,validation_language_coverage,True,missing languages with enough rows: []
6,noise_filter_removed_rows,True,removed 4467 of 4467 noise-hit rows
7,hard_noise_patterns_absent,True,remaining hard-block rows: 0


✅ All validation checks passed for this Phase 3 variant.

## 3. Save Dataset Artifacts

In [7]:
write_jsonl(RUN_DIR / "train.jsonl", train_rows)
write_jsonl(RUN_DIR / "validation.jsonl", validation_rows)

manifest = {
    "variant_name": VARIANT_NAME,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_name": DATASET_NAME,
    "seed": SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "required_columns": REQUIRED_COLUMNS,
    "flags": {
        "apply_noise_filter": APPLY_NOISE_FILTER,
        "apply_balancing": APPLY_BALANCING,
        "apply_high_confidence": APPLY_HIGH_CONFIDENCE,
    },
    "rows": {
        "raw_loaded": len(raw_rows),
        "prepared": len(prepared_rows),
        "deduplicated": len(dedup_rows),
        "final_cleaned": len(cleaned_rows),
        "train": len(train_rows),
        "validation": len(validation_rows),
    },
    "summaries": {
        "before": before_summary,
        "after_dedup": after_dedup_summary,
        "after": after_summary,
        "train": train_summary,
        "validation": validation_summary,
    },
    "removal_counts": dict(removal_counts),
    "balance_details": balance_details,
}

cleaning_report = {
    "manifest": manifest,
    "validation_results": validation_results,
    "language_distribution_before": language_stats(prepared_rows).to_dict(orient="records"),
    "language_distribution_after": language_stats(cleaned_rows).to_dict(orient="records"),
    "removed_examples": removed_examples,
}

dataset_card = f"""# {VARIANT_NAME}

Cleaned variant of `{DATASET_NAME}` for Llama 3.2-1B continued-pretraining experiments.

## Purpose

{VARIANT_PURPOSE}

## Cleaning Flags

- Noise filter: `{APPLY_NOISE_FILTER}`
- Language balancing: `{APPLY_BALANCING}`
- High-confidence filtering: `{APPLY_HIGH_CONFIDENCE}`

## Output

- Train rows: `{len(train_rows):,}`
- Validation rows: `{len(validation_rows):,}`
- Total cleaned rows: `{len(cleaned_rows):,}`
- Total cleaned tokens: `{after_summary['tokens']:,}`

## Validation

All notebook assertions passed when this artifact was generated. See `cleaning_report.json` for the full checklist, removal counts, language distribution, and representative removed examples.

## Intended Use

Continued pretraining / domain adaptation experiments. This is not instruction/chat data.
"""

(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
(RUN_DIR / "cleaning_report.json").write_text(json.dumps(cleaning_report, indent=2, ensure_ascii=False), encoding="utf-8")
(RUN_DIR / "dataset_card.md").write_text(dataset_card, encoding="utf-8")

DRIVE_RUN_DIR = DRIVE_ROOT / VARIANT_NAME
if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(RUN_DIR, DRIVE_RUN_DIR)

print(f"Saved local artifacts to: {RUN_DIR}")
print(f"Copied Drive artifacts to: {DRIVE_RUN_DIR}")
for artifact in ["train.jsonl", "validation.jsonl", "manifest.json", "cleaning_report.json", "dataset_card.md"]:
    print(" -", DRIVE_RUN_DIR / artifact)

Saved local artifacts to: /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results/_working/cleaned-filtered-v1
Copied Drive artifacts to: /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results/cleaned-filtered-v1
 - /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results/cleaned-filtered-v1/train.jsonl
 - /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results/cleaned-filtered-v1/validation.jsonl
 - /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results/cleaned-filtered-v1/manifest.json
 - /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results/cleaned-filtered-v1/cleaning_report.json
 - /Users/zeraphim/Documents/Files/ThisIsZeraphim/PH-SapinSapin/Training/Llama-3.2 1B/Phases/Colab/results/cleaned-fil